In [1]:
import pandas as pd
df = pd.read_parquet('../data/processed/dataset_ml_final.parquet')


In [3]:
import urllib.request
import urllib.parse
import json
import pandas as pd

# Open-Meteo Archive API : on récupère en DAILY puis on agrège en mensuel
# Paris comme proxy France (89% des commandes GE)
params = {
    'latitude':   '48.85',
    'longitude':  '2.35',
    'start_date': '2020-01-01',
    'end_date':   '2025-12-31',
    'daily':      ['precipitation_sum', 'wind_speed_10m_max', 'temperature_2m_min'],
    'timezone':   'Europe/Paris',
}
url = 'https://archive-api.open-meteo.com/v1/archive?' + urllib.parse.urlencode(params, doseq=True)

with urllib.request.urlopen(url, timeout=30) as resp:
    data = json.loads(resp.read().decode())

# Construction du DataFrame daily
daily = data['daily']
meteo_daily = pd.DataFrame({
    'date':             pd.to_datetime(daily['time']),
    'pluie_mm':         daily['precipitation_sum'],
    'vent_max_kmh':     daily['wind_speed_10m_max'],
    'temp_min_celsius': daily['temperature_2m_min'],
})

# Agrégation mensuelle : somme pour la pluie, max pour le vent, min pour la temp
meteo_fr = (
    meteo_daily
    .set_index('date')
    .resample('MS')   # MS = début de mois
    .agg({
        'pluie_mm':         'sum',
        'vent_max_kmh':     'max',
        'temp_min_celsius': 'min',
    })
    .rename(columns={'pluie_mm': 'pluie_mm_mois'})
    .reset_index()
)

print('Météo mensuelle récupérée :')
print(meteo_fr.head())
print(f'\n{len(meteo_fr)} mois de données météo')

Météo mensuelle récupérée :
        date  pluie_mm_mois  vent_max_kmh  temp_min_celsius
0 2020-01-01           29.4          36.7              -3.2
1 2020-02-01          103.7          49.1              -1.3
2 2020-03-01           67.7          42.3              -0.6
3 2020-04-01           30.4          35.1              -0.9
4 2020-05-01           37.3          36.5               4.3

72 mois de données météo
